In [ ]:
#Conditional Chain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal
# from langchain_core.runnables import RunnableParallel

load_dotenv()

model = ChatOpenAI()

parser = StrOutputParser()


class Feedback(BaseModel):
    sentiment: Literal["Pos", "Neg"] = Field(
        description="Give me the sentiment of the feedback"
    )


parser2 = PydanticOutputParser(pydantic_object=Feedback)

prompt1 = PromptTemplate(
    template="Classfy the sentiment of feedback of the text into Positive and Negative \n {feedback} \n {format_instructions}",
    input_variables=["feedback"],
    partial_variables={"format_instructions": parser2.get_format_instructions()},
)


classifier_chain = prompt1 | model | parser2
result = classifier_chain.invoke(
    {"feedback": "The animation of black clover is not that good"}
)

print(result)


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.runnables import RunnableBranch, RunnableLambda

load_dotenv()
model = ChatOpenAI()

parser = StrOutputParser()


class Feedback(BaseModel):
    sentiment: Literal["Negative", "Positive"] = Field(
        description="Give the sentiment of feedback"
    )


parser2 = PydanticOutputParser(pydantic_object=Feedback)

prompt1 = PromptTemplate(
    template="classify the sentiment of the {feedback} into text  \n {format_instructions}",
    input_variables=["feedback"],
    partial_variables={"format_instructions": parser2.get_format_instructions()},
)
classify_chain = prompt1 | model | parser2

prompt2 = PromptTemplate(
    template="write me an appropriate response to this Positive feedback \n {feedback}",
    input_variables=["feedback"],
)
prompt3 = PromptTemplate(
    template="write me an appropriate response to this Negative feedback\n{feedback}",
    input_variables=["feedback"],
)


branch_chain = RunnableBranch(
    (lambda x: x.sentiment == "Positive", prompt2 | model | parser),
    (lambda x: x.sentiment == "Negative", prompt3 | model | parser),
    RunnableLambda(lambda x: "Could not find the sentiment"),
)
chain = classify_chain | branch_chain
result = chain.invoke({"feedback": "The deforestation is terrible thing"})

print(result)


In [ ]:
# Parallel Chain
from langchain_openai import ChatOpenAI
# from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

model1 = ChatOpenAI()
# model2 = ChatAnthropic(model_name= 'claude opus 4.7')

prompt1 = PromptTemplate(
    template="Generate the detailed notes of text {text}", input_variables=["text"]
)
prompt2 = PromptTemplate(
    template="write me 5 questions of notes for quiz \n from {text}",
    input_variables=["text"],
)
prompt3 = PromptTemplate(
    template="Merege the notes and 5 questions of quiz{notes} -> & \n {quiz}",
    input_variables=["notes", "quiz"],
)

parser = StrOutputParser()

parallel_chain = RunnableParallel(
    {"notes": prompt1 | model1 | parser, "quiz": prompt2 | model2 | parser}
)

merge_chain = prompt3 | model1 | parser

chain = parallel_chain | merge_chain

text = """ Agentic AI is a type of AI system that can autonomously make decisions, plan actions and execute tasks to achieve specific goals with minimal human intervention. It focuses on goal-driven behavior, reasoning and interaction with tools and environments.

Specialised mastery: It’s trained and fine-tuned to handle a particular type of problem with great accuracy.
Tool usage: It can connect with and use specific tools like software, APIs, databases, etc to achieve its goal.
Goal-oriented actions: Instead of just giving information, it actively takes steps toward completing the task.
Efficient problem-solving: Because it can plan, adapt and take actions autonomously, it can handle tasks more efficiently in dynamic environments.
Traditional vs. Agentic AI: Unlike traditional AI systems that primarily respond to inputs, Agentic AI focuses on autonomous decision-making and goal-driven actions. It is defined by behavior (how it acts), not by the breadth of knowledge like General AI."""

result = chain.invoke({"text": text})
print(result)


In [ ]:
# Sequential Chain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
model = ChatOpenAI()

prompt1 = PromptTemplate(
    template="Generate a detailed report on {topic}", input_variables=["topic"]
)

prompt2 = PromptTemplate(
    template="Write me 5 facts from detailed report  \n{text}", input_variables=["text"]
)

parser = StrOutputParser()

chain = prompt1 | model | parser | prompt2 | model | parser

result = chain.invoke({"topic": "Gold Price in india"})
print(result)
chain.get_graph().print_ascii()
